# Chapter 2 — Vision → Language Projector

This notebook defines and validates the **VLM projector**.

Goal:
    map vision encoder embeddings → LM embedding space

This module:
- does NOT tokenize images
- does NOT run the language model
- only aligns representation geometry

## Why a Projector?

Vision encoders and language models live in different vector spaces.

Examples:
- Vision encoder: D_v = 768 / 1024 / 1152
- Language model: D_lm = 768 / 1024 / 2048

The projector enforces:
    ℝ^{D_v} → ℝ^{D_lm}

This is a learnable linear interface.

## Projector Variants

1. Linear:
    y = Wx + b

2. MLP:
    y = W2 σ(W1 x)

Tradeoff:
- Linear → stable, cheap, common (LLaVA-style)
- MLP → more expressive, riskier


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## Dimension Contract

The projector must satisfy:

    vision_embed_dim → lm_embed_dim

These values must match:
- vision encoder output
- LM token embedding size

In [2]:
VISION_DIM = 1024   # example: ViT-L / SigLIP
LM_DIM = 2048       # example: nanochat_vlm

BATCH = 2
NUM_PATCHES = 256

## Linear Projector

The simplest and most stable choice.

In [3]:
class LinearProjector(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.proj = nn.Linear(in_dim, out_dim)

    def forward(self, x):
        return self.proj(x)

## MLP Projector

Adds nonlinearity at the cost of stability.

In [4]:
class MLPProjector(nn.Module):
    def __init__(self, in_dim, out_dim, hidden_dim=None):
        super().__init__()
        hidden_dim = hidden_dim or out_dim
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x):
        return self.net(x)

## Fake Vision Embeddings

We simulate vision encoder output:
    (B, N_patches, D_v)

In [5]:
vision_feats = torch.randn(BATCH, NUM_PATCHES, VISION_DIM)

## Forward Pass Through Projector

In [6]:
proj = LinearProjector(VISION_DIM, LM_DIM)
lm_feats = proj(vision_feats)

print("Input shape:", vision_feats.shape)
print("Output shape:", lm_feats.shape)

Input shape: torch.Size([2, 256, 1024])
Output shape: torch.Size([2, 256, 2048])


## Sanity Check: Norms

Projection should not:
- collapse vectors
- explode magnitudes

In [7]:
in_norm = vision_feats.norm(dim=-1).mean()
out_norm = lm_feats.norm(dim=-1).mean()

print("Input norm:", in_norm.item())
print("Output norm:", out_norm.item())

Input norm: 31.922069549560547
Output norm: 26.07771873474121


## Sanity Check: Distribution

In [8]:
print("Input mean/std:",
      vision_feats.mean().item(),
      vision_feats.std().item())

print("Output mean/std:",
      lm_feats.mean().item(),
      lm_feats.std().item())

Input mean/std: 0.0010094188619405031 0.9978266358375549
Output mean/std: 0.0004403351340442896 0.5764579772949219


## Alignment with LM Token Embeddings

Projected vision features must live in the same scale
as LM token embeddings.

In [9]:
lm_token_embeds = torch.randn(BATCH, 16, LM_DIM)

print("LM token norm:",
      lm_token_embeds.norm(dim=-1).mean().item())
print("Vision-projected norm:",
      lm_feats.norm(dim=-1).mean().item())

LM token norm: 45.357757568359375
Vision-projected norm: 26.07771873474121


## Optional: Output Normalization

Some VLMs apply LayerNorm after projection.

In [10]:
class NormedLinearProjector(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.proj = nn.Linear(in_dim, out_dim)
        self.ln = nn.LayerNorm(out_dim)

    def forward(self, x):
        return self.ln(self.proj(x))

## Boundary

This notebook guarantees:
- correct dimensional mapping
- reasonable vector geometry
- safe integration with LM attention

It does NOT:
- decide where <im_patch> tokens go
- run attention
- train the projector